# Part 2 — Dispute model

Train on July + August (`dispute_raised` known), predict the probability of a dispute for every September shipment.

- Features: `src/features.py` — everything is known when the invoice goes out (shipment record, what was billed, and what the contract says it should cost from the Part 1 audit). No feature uses `dispute_raised`.
- Model and validation: `src/model.py` (L2 logistic regression).
- This notebook only calls those functions and answers the questions.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.audit import build_audit
from src.features import build_features
from src.model import (TARGET, C, fit, predict, score, time_split_scores, cv_scores, regularisation_scan,
                       calibration_table, coefficients, expected_log_loss, driver_tables, what_if_fixed,
                       write_predictions)

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

df = pd.read_csv(ROOT / 'data' / 'shipments.csv', parse_dates=['ship_date'])
audit = build_audit(df)
month = audit['ship_date'].dt.month
labelled = audit[month.isin([7, 8])]
september = audit[month == 9]
pd.DataFrame({'shipments': audit.groupby(month).size(), 'disputes': audit.groupby(month)[TARGET].sum(),
              'dispute_rate': audit.groupby(month)[TARGET].mean().round(3)})

,shipments,disputes,dispute_rate
ship_date,,,
7,2600,368.0,0.142
8,2600,417.0,0.160
9,1800,0.0,NaN


## Sanity checks

In [2]:
X_all = build_features(audit)
assert labelled[TARGET].notna().all() and september[TARGET].isna().all()
assert len(september) == 1800 and september['awb'].is_unique
assert np.isfinite(X_all.to_numpy(dtype=float)).all()
assert not any('dispute' in c for c in X_all.columns)
print(f'{X_all.shape[1]} features, all finite; no target in features; {len(labelled):,} labelled, {len(september):,} to score')
list(X_all.columns)

21 features, all finite; no target in features; 5,200 labelled, 1,800 to score


['is_overbilled',
 'log_overcharge_inr',
 'log_undercharge_inr',
 'log_billed_over_declared',
 'billed_minus_declared_kg',
 'log_vol_over_dead',
 'log_billed_kg',
 'is_cod',
 'log_billed_total',
 'merchant_M01',
 'merchant_M02',
 'merchant_M03',
 'merchant_M04',
 'merchant_M05',
 'merchant_M06',
 'merchant_M07',
 'merchant_M08',
 'merchant_M09',
 'merchant_M10',
 'merchant_M11',
 'merchant_M12']

## Q2. Validation approach and results

**Split.** September is a future month, so the main check is out-of-time: train on July, score August (and the reverse as a second check). Stratified 5-fold CV on Jul+Aug shows how stable the score is.

**Metric.** The deliverable is a probability, so the primary metric is **log loss** (rewards well-calibrated probabilities, punishes confident mistakes). Also reported: ROC AUC (ranking), PR AUC (finding the disputes; base rate ~15%), Brier score.

In [3]:
splits = time_split_scores(labelled)
splits.round(4)

,log_loss,brier,roc_auc,pr_auc,mean_prob,actual_rate
baseline: Jul rate -> Aug,0.4417,0.1350,0.5000,0.1604,0.1415,0.1604
model: train Jul -> score Aug,0.3779,0.1130,0.7329,0.4527,0.1463,0.1604
model: train Aug -> score Jul,0.3633,0.1062,0.7014,0.3828,0.1589,0.1415


In [4]:
cv = cv_scores(labelled)
cv.agg(['mean', 'std']).round(4)

,log_loss,brier,roc_auc,pr_auc,mean_prob,actual_rate
mean,0.3652,0.1082,0.7310,0.4263,0.1509,0.151
std,0.0064,0.0021,0.0148,0.0130,0.0042,0.000


How regularisation strength `C` was picked (Jul → Aug). The score is flat across C, so the choice is not fragile:

In [5]:
regularisation_scan(labelled).round(4)[['log_loss', 'roc_auc']]

,log_loss,roc_auc
C,,
0.01,0.3786,0.7351
0.03,0.3779,0.7338
0.10,0.3779,0.7329
0.30,0.3780,0.7325
1.00,0.3782,0.7323


Calibration on August (trained on July): predicted vs actual dispute rate by predicted-risk quintile.

In [6]:
jul, aug = labelled[labelled['ship_date'].dt.month == 7], labelled[labelled['ship_date'].dt.month == 8]
calibration_table(aug[TARGET], predict(fit(jul), aug))

,predicted,actual,shipments
bin,,,
"(0.0229, 0.0676]",0.052950,0.051923,520
"(0.0676, 0.0935]",0.080272,0.084615,520
"(0.0935, 0.124]",0.108911,0.128846,520
"(0.124, 0.17]",0.144322,0.163462,520
"(0.17, 0.875]",0.345219,0.373077,520


## Q1. September predictions
Final model trained on all of July + August, then every September shipment scored.

In [7]:
model = fit(labelled)
sep_prob = predict(model, september)
preds = write_predictions(september, sep_prob, ROOT / 'outputs' / 'september_predictions.csv')
check = pd.read_csv(ROOT / 'outputs' / 'september_predictions.csv')
assert list(check.columns) == ['awb', 'prob'] and len(check) == 1800
assert set(check['awb']) == set(september['awb']) and check['prob'].between(0, 1).all()
print(f"Wrote outputs/september_predictions.csv: {len(check):,} rows, header awb,prob, "
      f"prob {check['prob'].min():.3f}-{check['prob'].max():.3f}, mean {check['prob'].mean():.3f}")
check.head()

Wrote outputs/september_predictions.csv: 1,800 rows, header awb,prob, prob 0.027-0.958, mean 0.195


,awb,prob
0,SFX2704695529,0.189256
1,SFX2704800258,0.191691
2,SFX2704904987,0.323094
3,SFX2705009716,0.119120
4,SFX2705114445,0.039800


In [8]:
check['prob'].describe().round(3)

count    1800.000
mean        0.195
std         0.203
min         0.027
25%         0.075
50%         0.118
75%         0.192
max         0.958
Name: prob, dtype: float64

September differs from training: more shipments are overcharged (Part 1 C7: M09 billed at 8% instead of 12% on every September shipment). The model scores them through the general *overcharged* signal, not a zone-specific one:

In [9]:
sep_view = september.assign(prob=sep_prob.values)
sep_view.groupby('status')['prob'].agg(shipments='size', mean_prob='mean').round(3)

,shipments,mean_prob
status,,
ok,1180,0.119
overbilled,242,0.667
underbilled,378,0.126


In [10]:
print('Share of shipments overcharged:',
      f"Jul+Aug {(labelled['status'] == 'overbilled').mean():.1%}, Sep {(september['status'] == 'overbilled').mean():.1%}")
m09 = sep_view[(sep_view['merchant_id'] == 'M09') & (sep_view['status'] == 'overbilled')]
train_over = labelled[labelled['status'] == 'overbilled']
print(f"M09 Sep overcharged: {len(m09)} shipments, mean overcharge ₹{m09['diff_total'].mean():.2f}, mean prob {m09['prob'].mean():.3f}")
print(f"Training overcharges: smallest ₹{train_over['diff_total'].min():.2f}, "
      f"dispute rate {train_over[TARGET].mean():.1%} (flat across size quartiles)")

Share of shipments overcharged: Jul+Aug 6.0%, Sep 13.4%
M09 Sep overcharged: 139 shipments, mean overcharge ₹5.18, mean prob 0.669
Training overcharges: smallest ₹6.30, dispute rate 64.1% (flat across size quartiles)


## Q3. Features and model choice

### What drives disputes (raw Jul+Aug rates, not model output)

In [11]:
drivers = driver_tables(labelled)
for name, table in drivers.items():
    print(f'--- {name}')
    print(table, end='\n\n')

--- billing vs contract
                  shipments  dispute_rate
billing                                  
billed correctly       3773         0.118
overcharged             312         0.641
undercharged           1115         0.125

--- zone error: overcharge vs not
                              shipments  dispute_rate
zone                                                 
zone correct                       4711         0.118
zone wrong, net overcharged         312         0.641
zone wrong, net undercharged        177         0.158

--- billed / declared weight (correctly billed only)
                      shipments  dispute_rate
billed_over_declared                         
(0.999, 1.25]               793         0.088
(1.25, 1.493]               727         0.095
(1.493, 1.807]              748         0.118
(1.807, 2.273]              761         0.125
(2.273, 11.735]             744         0.167

--- payment mode
              shipments  dispute_rate
payment_mode                 

### Model view: standardised coefficients (+ raises dispute odds, per 1 std dev)

In [12]:
coefs = coefficients(model)
coefs.round(3).to_frame('coefficient')

,coefficient
merchant_M12,-0.190
merchant_M10,-0.128
merchant_M02,-0.101
merchant_M01,-0.095
merchant_M05,-0.093
log_billed_total,-0.040
merchant_M06,-0.025
merchant_M08,0.003
log_undercharge_inr,0.004
log_overcharge_inr,0.006


### What fixing billing would change in September (model what-if)
Re-score September as if every shipment had been billed exactly per contract.

In [13]:
what_if = what_if_fixed(model, september)
what_if

,shipments,expected_disputes_as_billed,expected_disputes_if_fixed,avoided
status,,,,
ok,1180.0,141.0,141.0,0.0
overbilled,242.0,161.3,39.3,122.0
underbilled,378.0,47.8,61.1,-13.3
total,1800.0,350.1,241.4,108.7


### Model choice

In [14]:
import lightgbm as lgb
from sklearn.metrics import log_loss, roc_auc_score
Xj, Xa = build_features(jul), build_features(aug)
gbm = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=80,
                         reg_lambda=5, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                         random_state=0, verbose=-1).fit(Xj, jul[TARGET])
p_gbm = gbm.predict_proba(Xa)[:, 1]
p_lr = predict(fit(jul), aug)
pd.DataFrame({'log_loss': [log_loss(aug[TARGET], p_lr), log_loss(aug[TARGET], p_gbm)],
              'roc_auc': [roc_auc_score(aug[TARGET], p_lr), roc_auc_score(aug[TARGET], p_gbm)]},
             index=['logistic regression', 'LightGBM (small, regularised)']).round(4)

,log_loss,roc_auc
logistic regression,0.3779,0.7329
"LightGBM (small, regularised)",0.3873,0.7127


## PART 2 ANSWERS (copy-paste)

In [15]:
s = splits
fwd, rev = s.loc['model: train Jul -> score Aug'], s.loc['model: train Aug -> score Jul']
base = s.loc['baseline: Jul rate -> Aug']
cvm, cvs = cv.mean(), cv.std()
exp_ll = expected_log_loss(sep_prob)
ll_lo, ll_hi = min(fwd['log_loss'], rev['log_loss'], cvm['log_loss'], exp_ll), max(fwd['log_loss'], rev['log_loss'], cvm['log_loss'], exp_ll)
auc_lo, auc_hi = min(fwd['roc_auc'], rev['roc_auc'], cvm['roc_auc']), max(fwd['roc_auc'], rev['roc_auc'], cvm['roc_auc'])
d = drivers
bill = d['billing vs contract']['dispute_rate']
zone = d['zone error: overcharge vs not']['dispute_rate']
wt = d['billed / declared weight (correctly billed only)']['dispute_rate']
pay = d['payment mode']['dispute_rate']
mer = d['merchant']['dispute_rate']
wi = what_if.loc['total']
wi_over, wi_under = what_if.loc['overbilled'], what_if.loc['underbilled']
sep_over = september[september['status'] == 'overbilled']
over_zone = (sep_over['billed_zone'] != sep_over['exp_zone']).sum()
over_m09 = sep_over['reason'].str.contains('old discount').sum()
assert over_zone + over_m09 == len(sep_over)
cal = calibration_table(aug[TARGET], predict(fit(jul), aug))
cal_gap = (cal['predicted'] - cal['actual']).abs().max()
lgb_ll = log_loss(aug[TARGET], p_gbm)

print('PART 2 ANSWERS')
print('=' * 70)
print(f"Q1. outputs/september_predictions.csv: header awb,prob, {len(preds):,} rows (one per September shipment), "
      f"prob {preds['prob'].min():.3f}-{preds['prob'].max():.3f}, mean {preds['prob'].mean():.3f}.")
print('\nQ2. Validation approach and results')
for line in [
    f"Split: out-of-time, because September is a future month: train July -> score August, plus the reverse (Aug -> Jul) "
    f"and stratified 5-fold CV on Jul+Aug for stability.",
    f"Metric: log loss (primary), because the deliverable is a probability and log loss rewards calibration; "
    f"ROC AUC and PR AUC for ranking, Brier as a second calibration check.",
    f"Jul -> Aug: log loss {fwd['log_loss']:.3f} vs {base['log_loss']:.3f} for a constant-rate baseline; "
    f"ROC AUC {fwd['roc_auc']:.3f}, PR AUC {fwd['pr_auc']:.3f} (base rate {fwd['actual_rate']:.1%}).",
    f"Aug -> Jul: log loss {rev['log_loss']:.3f}, ROC AUC {rev['roc_auc']:.3f}; 5-fold CV: log loss "
    f"{cvm['log_loss']:.3f} ± {cvs['log_loss']:.3f}, ROC AUC {cvm['roc_auc']:.3f} ± {cvs['roc_auc']:.3f}.",
    f"Calibration on August: predicted vs actual dispute rate within {cal_gap:.1%} in every risk quintile.",
    f"Expected on September: log loss about {ll_lo:.2f}-{ll_hi:.2f} and ROC AUC about {auc_lo:.2f}-{auc_hi:.2f}; "
    f"mean predicted dispute rate {sep_prob.mean():.1%} (vs {labelled[TARGET].mean():.1%} in Jul+Aug) because "
    f"{(september['status'] == 'overbilled').mean():.1%} of September shipments are overcharged vs "
    f"{(labelled['status'] == 'overbilled').mean():.1%} before.",
]:
    print('  - ' + line)
print('\nQ3. Features and model choice')
for line in [
    f"Biggest driver is being overcharged vs contract: {bill['overcharged']:.0%} dispute rate vs "
    f"{bill['billed correctly']:.0%} when billed correctly and {bill['undercharged']:.0%} when undercharged.",
    f"It is the overcharge, not the error itself: wrong-zone shipments that were net overcharged are disputed "
    f"{zone['zone wrong, net overcharged']:.0%}, but only {zone['zone wrong, net undercharged']:.0%} when other errors made them net undercharged.",
    f"Billed weight far above declared weight raises disputes even when billing is correct: "
    f"{wt.iloc[0]:.0%} in the lowest fifth of billed/declared vs {wt.iloc[-1]:.0%} in the highest.",
    f"COD shipments {pay['COD']:.0%} vs prepaid {pay['Prepaid']:.0%}; merchants differ a lot "
    f"({mer.index[0]} {mer.iloc[0]:.0%} to {mer.index[-1]} {mer.iloc[-1]:.0%}).",
    f"Model: L2 logistic regression on {X_all.shape[1]} features (billing error vs contract, weight gaps, invoice size, "
    f"COD, merchant); chosen over LightGBM (Aug log loss {fwd['log_loss']:.3f} vs {lgb_ll:.3f}) because it validated "
    f"better on ~2,600 rows a month, is well calibrated, and each driver can be read off its coefficient.",
    f"Ops: fix the overcharges first - the Hyderabad metro zone rule ({over_zone}) and the M09 12% discount "
    f"({over_m09}) cause all {wi_over['shipments']:.0f} September overcharges; billing them correctly cuts their expected disputes from "
    f"{wi_over['expected_disputes_as_billed']:.0f} to {wi_over['expected_disputes_if_fixed']:.0f}.",
    f"Ops: fixing the undercharges recovers revenue but adds about {-wi_under['avoided']:.0f} expected disputes "
    f"(heavier billed weights), so roll those fixes out with clear re-measure evidence on the invoice.",
    f"Ops: when billed weight is well above declared, send the re-measure photo/dimensions with the invoice, "
    f"and work with high-dispute merchants ({', '.join(mer.index[-3:][::-1])}) on declaring weights and dimensions accurately.",
]:
    print('  - ' + line)
print('=' * 70)

PART 2 ANSWERS
Q1. outputs/september_predictions.csv: header awb,prob, 1,800 rows (one per September shipment), prob 0.027-0.958, mean 0.195.

Q2. Validation approach and results
  - Split: out-of-time, because September is a future month: train July -> score August, plus the reverse (Aug -> Jul) and stratified 5-fold CV on Jul+Aug for stability.
  - Metric: log loss (primary), because the deliverable is a probability and log loss rewards calibration; ROC AUC and PR AUC for ranking, Brier as a second calibration check.
  - Jul -> Aug: log loss 0.378 vs 0.442 for a constant-rate baseline; ROC AUC 0.733, PR AUC 0.453 (base rate 16.0%).
  - Aug -> Jul: log loss 0.363, ROC AUC 0.701; 5-fold CV: log loss 0.365 ± 0.006, ROC AUC 0.731 ± 0.015.
  - Calibration on August: predicted vs actual dispute rate within 2.8% in every risk quintile.
  - Expected on September: log loss about 0.36-0.38 and ROC AUC about 0.70-0.73; mean predicted dispute rate 19.5% (vs 15.1% in Jul+Aug) because 13.4% of Sep